<a href="https://colab.research.google.com/github/Adrianoglima22/Agno-Examples/blob/main/Team_agno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [21]:
#!pip install -U agno openai tavily-python wikipedia plotly

In [22]:

import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")


# 1 — Entendendo a necessidade

Pergunta para o Treinador:

"Faça uma análise pré-jogo do próximo confronto da Seleção: quem está em forma, quem é o adversário, qual seria a estratégia tática?"

Três aspectos da pergunta:

levantamento de dados (forma + adversário),
análise tática (estratégia)
síntese (resposta organizada).

In [4]:

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_treinador = OpenAIChat(id="gpt-5.4-nano")

treinador_solo = Agent(
    name="Treinador",
    description="Assistente do CanarIA sobre a Seleção Brasileira masculina de futebol.",
    model= modelo_treinador,
    instructions=[
        "Você é o Treinador, assistente do CanarIA dedicado à Seleção Brasileira masculina.",
        "Responda em português do Brasil, com tom profissional e analítico.",
        "Quando não tiver certeza de um dado, diga claramente.",

        "POLÍTICA DE FONTES — siga rigorosamente:",
        "• Para EVENTOS RECENTES: use Tavily (busca web).",
        "• Para FATOS HISTÓRICOS CONSOLIDADOS: use Wikipedia.",
        "• Para PERGUNTAS CONCEITUAIS: responda direto sem tool.",
        "• Quando a pergunta tiver MÚLTIPLAS NECESSIDADES, combine fontes.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

treinador_solo.print_response(
    "Faça uma análise pré-jogo do próximo confronto da Seleção: "
    "quem está em forma, quem é o adversário, qual seria a estratégia tática?",
    stream=True,
)

Output()

O que dá pra observar:

1.   Os papéis estão misturados.
2.   Difícil melhorar uma parte sem afetar as outras.
3.    Difícil escalar.




# 2 — Time de agentes

```
      Pergunta do usuário
                   │
                   ▼
            Treinador (líder)
                   │
        ┌──────────┴──────────┐
        ▼                     ▼
    Olheiro              
   (busca dados)          Analista,
                        que interpreta
        │                     │

        └──────────┬──────────┘
                   ▼
            Treinador integra
                   │
                   ▼
              Resposta final
```

| Conceito do Agno | Equivalente no futebol |
|------------------|------------------------|
| Team | A comissão técnica como um todo |
| members | Os especialistas (Olheiro, Analista) |
| mode="coordinate" | O Treinador decide quem chamar e quando |
| instructions do Team | A "filosofia de trabalho" que o Treinador segue como líder |
| role de cada membro | O cargo formal de cada especialista |

# tools customizadas
Tool customizada no Agno é uma função Python decorada com @tool. A description é o que o LLM lê para decidir quando chamar (então ela tem que ser clara e específica).

In [25]:
import plotly.graph_objects as go
from agno.tools import tool

#Gera gráfico de barras

@tool(
    name="comparar_em_barras",
    description=(
        "Gera gráfico de barras verticais comparando jogadores em uma métrica única. "
        "Use quando o usuário pedir comparação direta entre jogadores em um único atributo "
        "(ex: gols, assistências, finalizações). "
        "Retorna confirmação de que o gráfico foi exibido."
    ),
)
def comparar_em_barras(
    jogadores: list[str],
    valores: list[float],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de barras interativo comparando jogadores."""
    fig = go.Figure(data=[go.Bar(x=jogadores, y=valores)])
    fig.update_layout(
        title=titulo,
        xaxis_title="Jogador",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de barras '{titulo}' exibido para o usuário."


In [26]:
#Gera gráfico de linhas

@tool(
    name="evolucao_em_linhas",
    description=(
        "Gera gráfico de linhas mostrando a evolução de jogadores ao longo do tempo. "
        "Use quando o usuário pedir progressão temporal "
        "(ex: 'como evoluiu nas últimas temporadas', 'desempenho ao longo dos anos'). "
        "Cada jogador é uma linha separada, períodos no eixo X. "
        "Retorna confirmação de que o gráfico foi exibido."
    ),
)
def evolucao_em_linhas(
    jogadores: list[str],
    periodos: list[str],
    valores_por_jogador: list[list[float]],
    metrica: str,
    titulo: str,
) -> str:
    """Gera gráfico de linhas interativo mostrando evolução temporal."""
    fig = go.Figure()
    for nome, valores in zip(jogadores, valores_por_jogador):
        fig.add_trace(go.Scatter(x=periodos, y=valores, mode="lines+markers", name=nome))
    fig.update_layout(
        title=titulo,
        xaxis_title="Período",
        yaxis_title=metrica.capitalize(),
        template="plotly_white",
        height=400,
    )
    fig.show()
    return f"Gráfico de linhas '{titulo}' exibido para o usuário."


# 3 — Olheiro

In [32]:
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

modelo_olheiro = OpenAIChat(id="gpt-5.4-nano")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas (web e Wikipedia)",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do CanarIA. Sua função é buscar informação verificável em fontes externas.",

        "POLÍTICA DE FONTES — siga rigorosamente:",
        "• Para EVENTOS RECENTES (últimos jogos, convocações, lesões, forma atual): "
        "use Tavily (busca web) como primeira opção.",
        "• Para FATOS HISTÓRICOS CONSOLIDADOS (Copas antigas, biografias, técnicos do passado, "
        "regulamentos): use Wikipedia — é mais estruturada e citável.",
        "Em caso de dúvida, prefira Tavily.",
        "Retorne dados objetivos — números, datas, nomes, fatos. Não interprete nem opine.",
        "Se a busca falhar, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    markdown=True,
)

# 4 — Analista

In [31]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat

modelo_analista = OpenAIChat(id="gpt-5.4-nano")

analista = Agent(
    name="Analista",
    role="Analista de desempenho que interpreta dados e gera visualizações táticas",
    model=modelo_analista,
    instructions=[
        "Você é o Analista de desempenho do CanarIA.",
        "Sua função é receber dados (geralmente do Olheiro) e produzir análise visual.",
        "Use `comparar_em_barras` para comparações diretas entre jogadores em uma métrica.",
        "Use `evolucao_em_linhas` para progressões ao longo do tempo.",
        "Sempre que gerar um gráfico, acompanhe de uma análise interpretativa de 2-3 frases "
        "destacando o que o gráfico revela.",
        "Quando receber pedido de gráfico ou visualização, "
        "SEMPRE chame `comparar_em_barras` ou `evolucao_em_linhas` — nunca produza ASCII, "
        "Mermaid ou tabelas como substituto. As tools são o entregável correto.",
        "Se receber dados insuficientes ou ambíguos, peça esclarecimento em vez de inventar números.",
    ],
    tools=[comparar_em_barras, evolucao_em_linhas],
    markdown=True,
)

Utilizando a classe **Team** do Agno.

In [36]:
from agno.team import Team

modelo_treinador = OpenAIChat(id="gpt-5.4-nano")

time_scoutai = Team(
    name="Treinador do CanarIA",
    mode="coordinate",
    members=[olheiro, analista],
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do CanarIA, líder da comissão técnica da seleção brasileira.",
        "Você coordena o Olheiro (busca dados) e o Analista (interpreta dados e gera gráficos).",

        "Quando o usuário pedir dados ou fatos verificáveis, delegue ao Olheiro.",
        "Quando o usuário pedir comparação visual, gráfico ou análise de evolução: "
        "primeiro garanta que o Olheiro trouxe os dados. Depois delegue ao Analista APENAS pedindo o gráfico",
        "o Analista tem tools próprias de Plotly e escolherá a apropriada.",
        "NÃO especifique formato (Mermaid, ASCII, tabela) — confie nas tools do Analista.",
        "Quando o usuário pedir conceito tático ou opinião, responda direto sem delegar.",
        "Sempre integre dados e análise antes de responder ao usuário, em português do Brasil.",
    ],
    markdown=True,
)


Agora montamos o time com três agentes:

1.   Treinador-líder (coordena e responde).
2.   Olheiro (busca),
3.   Analista (visualiza),

In [35]:
time_scoutai.print_response(
    "Compare os gols pela Seleção dos jogadores Vinícius Júnior, Rodrygo e Raphinha."
    "Gere um gráfico para ilustrar essa comparação.",
    stream=True,
)

Output()

INFO Searching wikipedia for: Vinícius Júnior Seleção Brasileira  gols 13 2026 Wikipédia

ERROR    Error searching Wikipedia for 'Vinícius Júnior Seleção Brasileira  gols 13 2026 Wikipédia': Expecting     
         value: line 1 column 1 (char 0)

INFO Searching wikipedia for: Rodrygo Seleção Brasileira  gols 9 Wikipédia

ERROR    Error searching Wikipedia for 'Rodrygo Seleção Brasileira  gols 9 Wikipédia': Expecting value: line 1     
         column 1 (char 0)

INFO Searching wikipedia for: Raphinha Seleção Brasileira  gols 11 Wikipédia

ERROR    Error searching Wikipedia for 'Raphinha Seleção Brasileira  gols 11 Wikipédia': Expecting value: line 1   
         column 1 (char 0)

In [39]:
time_scoutai.print_response(
    "Faz uma análise pré-jogo do próximo confronto da Seleção: "
    "quem está em forma, quem é o adversário, qual seria a estratégia tática?",
    stream=True,
)

Output()